In [ ]:
print("OM SAI RAM !!")

## Explore the dataset

In [ ]:
import os

DATASET_ROOT = "/kaggle/input"

print("=" * 75)
print("AVAILABLE KAGGLE DATASETS")
print("=" * 75)

for root, dirs, files in os.walk(DATASET_ROOT):
    depth = root.replace(DATASET_ROOT, "").count(os.sep)

    if depth > 3:
        dirs[:] = []
        continue

    indent = "    " * depth
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:5]:
        print(f"{indent}    {file}")

    if len(files) > 5:
        print(f"{indent}    ... and {len(files) - 5} more files")

## Inspect the structures of all the three datasets

In [ ]:
import os

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

DATASETS = {
    "Dataset 1": os.path.join(BASE_PATH, "dataset1"),
    "Dataset 2": os.path.join(BASE_PATH, "dataset2"),
    "Mendeley": os.path.join(BASE_PATH, "mendely-dataset")
}

for dataset_name, dataset_path in DATASETS.items():

    print("\n" + "=" * 80)
    print(f"{dataset_name.upper()}")
    print(f"Path: {dataset_path}")
    print("=" * 80)

    if not os.path.exists(dataset_path):
        print("DATASET PATH NOT FOUND!")
        continue

    for root, dirs, files in os.walk(dataset_path):
        depth = root.replace(dataset_path, "").count(os.sep)

        if depth > 3:
            dirs[:] = []
            continue

        indent = "    " * depth
        print(f"{indent}{os.path.basename(root)}/")

        for file in files[:8]:
            print(f"{indent}    {file}")

        if len(files) > 8:
            print(f"{indent}    ... and {len(files) - 8} more files")

## Count the number of images belonging to every class

In [ ]:
import os
from collections import defaultdict

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

DATASETS = {
    "Dataset 1": os.path.join(BASE_PATH, "dataset1", "dataset"),
    "Dataset 2": os.path.join(BASE_PATH, "dataset2", "dataset"),
    "Mendeley": os.path.join(BASE_PATH, "mendely-dataset", "Mendely")
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for dataset_name, dataset_path in DATASETS.items():

    print("\n" + "=" * 75)
    print(f"{dataset_name.upper()} - IMAGE COUNTS")
    print("=" * 75)

    for root, dirs, files in os.walk(dataset_path):
        image_files = [
            f for f in files
            if f.lower().endswith(IMAGE_EXTENSIONS)
        ]

        if image_files:
            relative_path = os.path.relpath(root, dataset_path)
            print(f"{relative_path:30s}: {len(image_files):5d}")

    print("-" * 75)

# Visual dataset auditing

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

DATASET_FOLDERS = {
    "Dataset 1 - Train": os.path.join(BASE_PATH, "dataset1", "dataset", "train"),
    "Dataset 1 - Test": os.path.join(BASE_PATH, "dataset1", "dataset", "test"),
    "Dataset 2 - Train": os.path.join(BASE_PATH, "dataset2", "dataset", "train"),
    "Dataset 2 - Test": os.path.join(BASE_PATH, "dataset2", "dataset", "test"),
    "Mendeley - Cropped": os.path.join(BASE_PATH, "mendely-dataset", "Mendely", "Cropped Image"),
    "Mendeley - Primary": os.path.join(BASE_PATH, "mendely-dataset", "Mendely", "Primary Image")
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

samples = []

for dataset_name, dataset_path in DATASET_FOLDERS.items():

    if not os.path.exists(dataset_path):
        continue

    for class_name in sorted(os.listdir(dataset_path)):

        class_path = os.path.join(dataset_path, class_name)

        if not os.path.isdir(class_path):
            continue

        image_files = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(IMAGE_EXTENSIONS)
        ]

        if not image_files:
            continue

        selected_file = random.choice(image_files)
        image_path = os.path.join(class_path, selected_file)

        try:
            with Image.open(image_path) as img:
                image_size = img.size
                image_mode = img.mode

                samples.append({
                    "dataset": dataset_name,
                    "class": class_name,
                    "path": image_path,
                    "size": image_size,
                    "mode": image_mode
                })

        except Exception as e:
            print(f"Could not read {image_path}: {e}")

print("=" * 90)
print("REPRESENTATIVE IMAGE DETAILS")
print("=" * 90)

for sample in samples:
    print(
        f"{sample['dataset']:24s} | "
        f"{sample['class']:12s} | "
        f"Size: {str(sample['size']):15s} | "
        f"Mode: {sample['mode']:5s} | "
        f"{os.path.basename(sample['path'])}"
    )

# Display representative images
columns = 4
rows = (len(samples) + columns - 1) // columns

plt.figure(figsize=(18, rows * 4))

for i, sample in enumerate(samples):

    with Image.open(sample["path"]) as img:
        img = img.convert("RGB")

        plt.subplot(rows, columns, i + 1)
        plt.imshow(img)
        plt.title(
            f"{sample['dataset']}\n"
            f"Class: {sample['class']}\n"
            f"Size: {sample['size']}",
            fontsize=9
        )
        plt.axis("off")

plt.tight_layout()
plt.show()

## Duplication and leakage auditing

In [ ]:
import os
import hashlib
from collections import defaultdict
from PIL import Image
from tqdm.auto import tqdm

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

FOLDERS = {
    "Dataset1": os.path.join(BASE_PATH, "dataset1", "dataset", "train"),
    "Dataset2": os.path.join(BASE_PATH, "dataset2", "dataset", "train"),
    "Mendeley_Cropped": os.path.join(BASE_PATH, "mendely-dataset", "Mendely", "Cropped Image"),
    "Mendeley_Primary": os.path.join(BASE_PATH, "mendely-dataset", "Mendely", "Primary Image")
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ---------------------------------------------------------
# 1. Collect all image paths
# ---------------------------------------------------------

print("=" * 85)
print("SCANNING DATASET DIRECTORIES")
print("=" * 85)

all_images = []

for dataset_name, dataset_path in FOLDERS.items():

    dataset_images = []

    for root, dirs, files in os.walk(dataset_path):
        for filename in files:
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                dataset_images.append(
                    (dataset_name, os.path.join(root, filename), dataset_path)
                )

    all_images.extend(dataset_images)

    print(f"{dataset_name:20s}: {len(dataset_images):5d} images found")

print("-" * 85)
print(f"TOTAL IMAGES TO AUDIT: {len(all_images)}")

# ---------------------------------------------------------
# 2. Exact duplicate detection using decoded RGB pixels
# ---------------------------------------------------------

print("\n" + "=" * 85)
print("EXACT DUPLICATE IMAGE DETECTION")
print("=" * 85)

hash_records = defaultdict(list)
error_count = 0

for dataset_name, image_path, dataset_path in tqdm(
    all_images,
    desc="Hashing images",
    unit="image",
    dynamic_ncols=True
):

    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            image_hash = hashlib.sha256(img.tobytes()).hexdigest()

        relative_path = os.path.relpath(image_path, dataset_path)

        hash_records[image_hash].append(
            (dataset_name, relative_path)
        )

    except Exception as e:
        error_count += 1
        tqdm.write(f"Error reading {image_path}: {e}")

duplicate_groups = [
    records for records in hash_records.values()
    if len(records) > 1
]

print("\n" + "-" * 85)
print("DUPLICATE DETECTION RESULTS")
print("-" * 85)

print("Total images scanned       :", len(all_images))
print("Successfully hashed images :", sum(len(v) for v in hash_records.values()))
print("Unique image hashes        :", len(hash_records))
print("Duplicate groups           :", len(duplicate_groups))
print("Unreadable images           :", error_count)

if duplicate_groups:

    print("\nDUPLICATE GROUP DETAILS")

    for index, records in enumerate(duplicate_groups, start=1):

        print(f"\nDuplicate Group {index}:")

        for dataset_name, filename in records:
            print(f"  {dataset_name:20s} | {filename}")

else:
    print("\nNo exact duplicate images found.")

# ---------------------------------------------------------
# 3. Mendeley primary vs cropped filename/index comparison
# ---------------------------------------------------------

print("\n" + "=" * 85)
print("MENDELEY PRIMARY VS CROPPED FILENAME AUDIT")
print("=" * 85)

for class_name in tqdm(
    ["Angry", "Cry", "Laugh", "Normal"],
    desc="Checking Mendeley classes",
    unit="class"
):

    cropped_dir = os.path.join(FOLDERS["Mendeley_Cropped"], class_name)
    primary_dir = os.path.join(FOLDERS["Mendeley_Primary"], class_name)

    cropped_files = sorted(os.listdir(cropped_dir))
    primary_files = sorted(os.listdir(primary_dir))

    print(f"\nCLASS: {class_name}")
    print("Cropped examples:", cropped_files[:8])
    print("Primary examples:", primary_files[:8])
    print("Cropped count   :", len(cropped_files))
    print("Primary count   :", len(primary_files))

print("\n" + "=" * 85)
print("AUDIT COMPLETED")
print("=" * 85)

## Exact duplicates summary

In [ ]:
import os
import hashlib
from collections import defaultdict
from PIL import Image
from tqdm.auto import tqdm

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

FOLDERS = {
    "Dataset1": os.path.join(BASE_PATH, "dataset1", "dataset", "train"),
    "Dataset2": os.path.join(BASE_PATH, "dataset2", "dataset", "train"),
    "Mendeley_Cropped": os.path.join(BASE_PATH, "mendely-dataset", "Mendely", "Cropped Image"),
    "Mendeley_Primary": os.path.join(BASE_PATH, "mendely-dataset", "Mendely", "Primary Image")
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Collect images
all_images = []

for dataset_name, dataset_path in FOLDERS.items():
    for root, dirs, files in os.walk(dataset_path):
        for filename in files:
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                all_images.append(
                    (dataset_name, os.path.join(root, filename))
                )

print(f"Total images found: {len(all_images)}")

# Hash images
hash_records = defaultdict(list)
errors = []

for dataset_name, image_path in tqdm(
    all_images,
    desc="Checking exact duplicates",
    unit="image"
):
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            digest = hashlib.sha256(img.tobytes()).hexdigest()

        hash_records[digest].append(
            (dataset_name, os.path.basename(image_path))
        )

    except Exception as e:
        errors.append((image_path, str(e)))

# Results
duplicates = [
    records for records in hash_records.values()
    if len(records) > 1
]

print("\n" + "=" * 75)
print("DUPLICATE AUDIT RESULTS")
print("=" * 75)

print(f"Total images scanned       : {len(all_images)}")
print(f"Unique image hashes        : {len(hash_records)}")
print(f"Duplicate groups           : {len(duplicates)}")
print(f"Images in duplicate groups : {sum(len(x) for x in duplicates)}")
print(f"Unreadable images          : {len(errors)}")

for i, group in enumerate(duplicates[:20], 1):
    print(f"\nDuplicate group {i}:")
    for dataset_name, filename in group:
        print(f"  {dataset_name:20s} | {filename}")

if not duplicates:
    print("\nNo exact pixel-level duplicates found.")

print("=" * 75)

## Complete duplicate analysis

In [ ]:
import os
import hashlib
from collections import defaultdict, Counter
from PIL import Image
from tqdm.auto import tqdm

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

FOLDERS = {
    "Dataset1": os.path.join(BASE_PATH, "dataset1", "dataset", "train"),
    "Dataset2": os.path.join(BASE_PATH, "dataset2", "dataset", "train"),
    "Mendeley_Cropped": os.path.join(BASE_PATH, "mendely-dataset", "Mendely", "Cropped Image"),
    "Mendeley_Primary": os.path.join(BASE_PATH, "mendely-dataset", "Mendely", "Primary Image")
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ---------------------------------------------------------
# 1. Recalculate hashes with class information
# ---------------------------------------------------------

hash_records = defaultdict(list)
all_images = []

for dataset_name, dataset_path in FOLDERS.items():

    for root, dirs, files in os.walk(dataset_path):

        for filename in files:

            if filename.lower().endswith(IMAGE_EXTENSIONS):
                all_images.append((dataset_name, dataset_path,
                                   root, filename))

for dataset_name, dataset_path, root, filename in tqdm(
    all_images,
    desc="Analyzing duplicate groups",
    unit="image"
):

    image_path = os.path.join(root, filename)
    class_name = os.path.basename(root)

    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            digest = hashlib.sha256(img.tobytes()).hexdigest()

        hash_records[digest].append({
            "dataset": dataset_name,
            "class": class_name,
            "filename": filename
        })

    except Exception as e:
        print(f"Error: {image_path} | {e}")

duplicates = [
    records for records in hash_records.values()
    if len(records) > 1
]

# ---------------------------------------------------------
# 2. Classify duplicate groups
# ---------------------------------------------------------

same_class_groups = []
cross_class_groups = []
cross_dataset_groups = []

class_pair_counts = Counter()
dataset_pair_counts = Counter()

for group in duplicates:

    classes = sorted(set(
        f"{record['dataset']}/{record['class']}"
        for record in group
    ))

    datasets = sorted(set(
        record["dataset"] for record in group
    ))

    if len(classes) == 1:
        same_class_groups.append(group)
    else:
        cross_class_groups.append(group)

        for i in range(len(classes)):
            for j in range(i + 1, len(classes)):
                class_pair_counts[(classes[i], classes[j])] += 1

    if len(datasets) > 1:
        cross_dataset_groups.append(group)

        for i in range(len(datasets)):
            for j in range(i + 1, len(datasets)):
                dataset_pair_counts[
                    (datasets[i], datasets[j])
                ] += 1

# ---------------------------------------------------------
# 3. Print summary
# ---------------------------------------------------------

print("\n" + "=" * 85)
print("COMPLETE DUPLICATE ANALYSIS")
print("=" * 85)

print(f"Total images scanned                 : {len(all_images)}")
print(f"Unique image hashes                  : {len(hash_records)}")
print(f"Total duplicate groups               : {len(duplicates)}")
print(f"Same-class duplicate groups          : {len(same_class_groups)}")
print(f"Cross-class duplicate groups         : {len(cross_class_groups)}")
print(f"Cross-dataset duplicate groups       : {len(cross_dataset_groups)}")

print("\n" + "-" * 85)
print("DUPLICATES BY CLASS COMBINATION")
print("-" * 85)

for pair, count in class_pair_counts.most_common():
    print(f"{pair[0]:35s} <-> {pair[1]:35s} : {count}")

print("\n" + "-" * 85)
print("DUPLICATES BY DATASET COMBINATION")
print("-" * 85)

for pair, count in dataset_pair_counts.most_common():
    print(f"{pair[0]:25s} <-> {pair[1]:25s} : {count}")

# ---------------------------------------------------------
# 4. Print every cross-class duplicate group
# ---------------------------------------------------------

print("\n" + "=" * 85)
print("ALL CROSS-CLASS DUPLICATE GROUPS")
print("=" * 85)

for index, group in enumerate(cross_class_groups, start=1):

    print(f"\nGroup {index}:")

    for record in group:
        print(
            f"  {record['dataset']:20s} | "
            f"{record['class']:12s} | "
            f"{record['filename']}"
        )

print("\n" + "=" * 85)
print("ANALYSIS COMPLETED")
print("=" * 85)

## Visual inspection of matching images

In [ ]:
import os
import hashlib
from collections import defaultdict, Counter
from PIL import Image
from tqdm.auto import tqdm

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

FOLDERS = {
    "Dataset1": os.path.join(BASE_PATH, "dataset1", "dataset", "train"),
    "Dataset2": os.path.join(BASE_PATH, "dataset2", "dataset", "train"),
    "Mendeley_Cropped": os.path.join(
        BASE_PATH, "mendely-dataset", "Mendely", "Cropped Image"
    )
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ---------------------------------------------------------
# STEP 1: DISCOVER ALL IMAGE FILES
# ---------------------------------------------------------

print("=" * 80)
print("STEP 1: DISCOVERING IMAGE FILES")
print("=" * 80)

all_image_paths = []

for dataset_name, dataset_path in FOLDERS.items():

    dataset_images = []

    for root, dirs, files in os.walk(dataset_path):
        for filename in files:
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                dataset_images.append({
                    "dataset": dataset_name,
                    "class": os.path.basename(root),
                    "filename": filename,
                    "path": os.path.join(root, filename)
                })

    all_image_paths.extend(dataset_images)

    print(f"{dataset_name}: {len(dataset_images)} images found")

print(f"\nTotal images to scan: {len(all_image_paths)}")

# ---------------------------------------------------------
# STEP 2: HASH ALL IMAGES
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 2: CALCULATING IMAGE HASHES")
print("=" * 80)

hash_records = defaultdict(list)
unreadable_images = []

for record in tqdm(
    all_image_paths,
    desc="Hashing images",
    unit="image"
):

    try:
        with Image.open(record["path"]) as img:
            digest = hashlib.sha256(
                img.convert("RGB").tobytes()
            ).hexdigest()

        hash_records[digest].append(record)

    except Exception as e:
        unreadable_images.append({
            "path": record["path"],
            "error": str(e)
        })

# ---------------------------------------------------------
# STEP 3: IDENTIFY LABEL CONFLICTS
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 3: IDENTIFYING LABEL CONFLICTS")
print("=" * 80)

conflicts = []

for digest, records in tqdm(
    hash_records.items(),
    desc="Checking duplicate groups",
    unit="group"
):

    labels = set(
        (r["dataset"], r["class"]) for r in records
    )

    if len(labels) > 1:
        conflicts.append(records)

# ---------------------------------------------------------
# STEP 4: CONFLICT SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("LABEL CONFLICT SUMMARY")
print("=" * 80)

pair_counts = Counter()

for group in tqdm(
    conflicts,
    desc="Summarizing conflicts",
    unit="group"
):

    labels = sorted(set(
        f"{r['dataset']}/{r['class']}" for r in group
    ))

    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            pair_counts[(labels[i], labels[j])] += 1

for pair, count in pair_counts.most_common():
    print(f"{pair[0]}  <-->  {pair[1]} : {count}")

# ---------------------------------------------------------
# STEP 5: DISPLAY FIRST 10 CONFLICTING GROUPS
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("FIRST 10 CONFLICTING GROUPS")
print("=" * 80)

for i, group in enumerate(conflicts[:10], 1):

    print(f"\nConflict {i}")

    for r in group:
        print(f"  Label: {r['dataset']}/{r['class']}")
        print(f"  File : {r['filename']}")

# ---------------------------------------------------------
# STEP 6: FINAL AUDIT SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL AUDIT SUMMARY")
print("=" * 80)

print(f"Total images discovered : {len(all_image_paths)}")
print(f"Successfully hashed     : {sum(len(v) for v in hash_records.values())}")
print(f"Unique image hashes     : {len(hash_records)}")
print(f"Conflicting groups      : {len(conflicts)}")
print(f"Unreadable images       : {len(unreadable_images)}")

print("\nAudit completed successfully.")

In [ ]:
import os
import hashlib
from collections import defaultdict, Counter
from PIL import Image
from tqdm.auto import tqdm

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

FOLDERS = {
    "Dataset1": os.path.join(BASE_PATH, "dataset1", "dataset", "train"),
    "Dataset2": os.path.join(BASE_PATH, "dataset2", "dataset", "train"),
    "Mendeley_Cropped": os.path.join(
        BASE_PATH, "mendely-dataset", "Mendely", "Cropped Image"
    )
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# =========================================================
# STEP 1: DISCOVER IMAGES
# =========================================================

print("=" * 80)
print("STEP 1: DISCOVERING IMAGES")
print("=" * 80)

all_image_paths = []

for dataset_name, dataset_path in FOLDERS.items():

    dataset_images = []

    for root, dirs, files in os.walk(dataset_path):
        for filename in files:

            if filename.lower().endswith(IMAGE_EXTENSIONS):
                dataset_images.append({
                    "dataset": dataset_name,
                    "class": os.path.basename(root),
                    "filename": filename,
                    "path": os.path.join(root, filename)
                })

    all_image_paths.extend(dataset_images)

    print(f"{dataset_name}: {len(dataset_images)} images")

print(f"\nTotal images discovered: {len(all_image_paths)}")

# =========================================================
# STEP 2: HASH IMAGES
# =========================================================

print("\n" + "=" * 80)
print("STEP 2: HASHING IMAGES")
print("=" * 80)

hash_records = defaultdict(list)
unreadable_images = []

for record in tqdm(
    all_image_paths,
    desc="Hashing images",
    unit="image"
):

    try:
        with Image.open(record["path"]) as img:

            digest = hashlib.sha256(
                img.convert("RGB").tobytes()
            ).hexdigest()

        hash_records[digest].append(record)

    except Exception as e:
        unreadable_images.append({
            "path": record["path"],
            "error": str(e)
        })

# =========================================================
# STEP 3: IDENTIFY CONFLICTING GROUPS
# =========================================================

print("\n" + "=" * 80)
print("STEP 3: IDENTIFYING LABEL CONFLICTS")
print("=" * 80)

conflicting_hashes = set()

for digest, records in tqdm(
    hash_records.items(),
    desc="Checking duplicate groups",
    unit="group"
):

    labels = {
        (r["dataset"], r["class"])
        for r in records
    }

    if len(labels) > 1:
        conflicting_hashes.add(digest)

conflicting_images = sum(
    len(hash_records[digest])
    for digest in conflicting_hashes
)

print(f"\nConflicting groups: {len(conflicting_hashes)}")
print(f"Images in conflicting groups: {conflicting_images}")

# =========================================================
# STEP 4: CREATE CLEAN COLLECTION
# =========================================================

print("\n" + "=" * 80)
print("STEP 4: FILTERING IMAGES")
print("=" * 80)

clean_records = []
removed_redundant_images = 0

for digest, records in tqdm(
    hash_records.items(),
    desc="Filtering image groups",
    unit="group"
):

    # Exclude all images from conflicting groups
    if digest in conflicting_hashes:
        continue

    # Retain one copy of each non-conflicting exact image
    clean_records.append(records[0])

    removed_redundant_images += len(records) - 1

# =========================================================
# STEP 5: CALCULATE CLEAN CLASS DISTRIBUTION
# =========================================================

print("\n" + "=" * 80)
print("STEP 5: CLEAN DATASET DISTRIBUTION")
print("=" * 80)

class_counts = defaultdict(Counter)

for record in tqdm(
    clean_records,
    desc="Counting clean images",
    unit="image"
):

    class_counts[record["dataset"]][record["class"]] += 1

for dataset_name in FOLDERS:

    print(f"\n{dataset_name}")

    total = 0

    for class_name, count in sorted(
        class_counts[dataset_name].items()
    ):

        print(f"  {class_name:<15}: {count}")
        total += count

    print(f"  {'TOTAL':<15}: {total}")

# =========================================================
# STEP 6: FINAL SUMMARY
# =========================================================

print("\n" + "=" * 80)
print("FINAL CLEANING SUMMARY")
print("=" * 80)

print(f"Images discovered              : {len(all_image_paths)}")
print(f"Successfully hashed            : {sum(len(v) for v in hash_records.values())}")
print(f"Unique image hashes            : {len(hash_records)}")
print(f"Conflicting groups             : {len(conflicting_hashes)}")
print(f"Images in conflicting groups   : {conflicting_images}")
print(f"Redundant copies removed       : {removed_redundant_images}")
print(f"Remaining clean images         : {len(clean_records)}")
print(f"Unreadable images              : {len(unreadable_images)}")

print("\nCleaning analysis completed.")

## Inspect dataset documentation

In [ ]:
import os
from tqdm.auto import tqdm

BASE_PATH = "/kaggle/input/datasets/prishabhkumar"

DATASET_ROOTS = {
    "Dataset1": os.path.join(BASE_PATH, "dataset1"),
    "Dataset2": os.path.join(BASE_PATH, "dataset2"),
    "Mendeley": os.path.join(BASE_PATH, "mendely-dataset")
}

DOCUMENT_EXTENSIONS = (
    ".txt", ".md", ".csv", ".json",
    ".xml", ".yaml", ".yml"
)

# ---------------------------------------------------------
# STEP 1: DISCOVER DOCUMENTATION FILES
# ---------------------------------------------------------

print("=" * 80)
print("SEARCHING FOR DATASET DOCUMENTATION")
print("=" * 80)

documentation_files = []

for dataset_name, dataset_root in DATASET_ROOTS.items():

    dataset_files = []

    for root, dirs, files in os.walk(dataset_root):
        for filename in files:

            if filename.lower().endswith(DOCUMENT_EXTENSIONS):
                dataset_files.append(
                    (dataset_name, os.path.join(root, filename))
                )

    documentation_files.extend(dataset_files)

    print(f"{dataset_name}: {len(dataset_files)} documentation files")

# ---------------------------------------------------------
# STEP 2: DISPLAY DOCUMENTATION CONTENT
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("DOCUMENTATION CONTENT")
print("=" * 80)

if not documentation_files:
    print("No documentation files found.")

for dataset_name, file_path in tqdm(
    documentation_files,
    desc="Reading documentation",
    unit="file"
):

    print("\n" + "-" * 80)
    print(f"DATASET: {dataset_name}")
    print(f"FILE: {os.path.relpath(file_path, BASE_PATH)}")
    print("-" * 80)

    try:
        with open(file_path, "r", encoding="utf-8", errors="replace") as f:
            content = f.read(12000)

        print(content if content.strip() else "[Empty file]")

    except Exception as e:
        print(f"Could not read file: {e}")

print("\nDocumentation inspection completed.")

In [ ]:
import os
import re
from collections import defaultdict
from tqdm.auto import tqdm

MENDELEY_ROOT = (
    "/kaggle/input/datasets/prishabhkumar/"
    "mendely-dataset/Mendely"
)

FOLDERS = {
    "Primary": os.path.join(MENDELEY_ROOT, "Primary Image"),
    "Cropped": os.path.join(MENDELEY_ROOT, "Cropped Image")
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ---------------------------------------------------------
# STEP 1: DISCOVER FILES
# ---------------------------------------------------------

print("=" * 80)
print("STEP 1: DISCOVERING MENDELEY IMAGE FILES")
print("=" * 80)

file_records = []

for representation, folder in FOLDERS.items():

    for root, dirs, files in os.walk(folder):
        for filename in files:

            if filename.lower().endswith(IMAGE_EXTENSIONS):
                file_records.append({
                    "representation": representation,
                    "class": os.path.basename(root).lower(),
                    "filename": filename,
                    "path": os.path.join(root, filename)
                })

print(f"Total image files discovered: {len(file_records)}")

# ---------------------------------------------------------
# STEP 2: EXTRACT CLASS AND IMAGE INDEX
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 2: EXTRACTING IMAGE INDICES")
print("=" * 80)

indexed_files = defaultdict(list)
unmatched_filenames = []

for record in tqdm(
    file_records,
    desc="Extracting filename indices",
    unit="file"
):

    match = re.search(r"\(\s*(\d+)\s*\)", record["filename"])

    if match:
        index = int(match.group(1))

        key = (
            record["representation"],
            record["class"],
            index
        )

        indexed_files[key].append(record)

    else:
        unmatched_filenames.append(record)

# ---------------------------------------------------------
# STEP 3: COMPARE PRIMARY AND CROPPED
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 3: PRIMARY-CROPPED PAIRING ANALYSIS")
print("=" * 80)

classes = ["angry", "cry", "laugh", "normal"]

for class_name in classes:

    primary_indices = {
        index for representation, cls, index in indexed_files
        if representation == "Primary" and cls == class_name
    }

    cropped_indices = {
        index for representation, cls, index in indexed_files
        if representation == "Cropped" and cls == class_name
    }

    common = primary_indices & cropped_indices
    primary_only = primary_indices - cropped_indices
    cropped_only = cropped_indices - primary_indices

    print(f"\nCLASS: {class_name.upper()}")
    print(f"Primary unique indices : {len(primary_indices)}")
    print(f"Cropped unique indices : {len(cropped_indices)}")
    print(f"Matching indices       : {len(common)}")
    print(f"Primary-only indices   : {len(primary_only)}")
    print(f"Cropped-only indices   : {len(cropped_only)}")

    if primary_only:
        print(f"  Primary-only examples: {sorted(primary_only)[:10]}")

    if cropped_only:
        print(f"  Cropped-only examples: {sorted(cropped_only)[:10]}")

# ---------------------------------------------------------
# STEP 4: FIND MULTIPLE FILES FOR THE SAME INDEX
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 4: MULTIPLE FILES PER IMAGE INDEX")
print("=" * 80)

multiple_files = []

for key, records in indexed_files.items():

    if len(records) > 1:
        multiple_files.append((key, records))

print(f"Indices having multiple files: {len(multiple_files)}")

for key, records in multiple_files[:20]:

    representation, class_name, index = key

    print(f"\n{representation}/{class_name} - Index {index}")

    for record in records:
        print(f"  {record['filename']}")

# ---------------------------------------------------------
# STEP 5: FINAL SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(f"Total files scanned       : {len(file_records)}")
print(f"Files with parsed indices: {len(file_records) - len(unmatched_filenames)}")
print(f"Unmatched filenames       : {len(unmatched_filenames)}")
print(f"Indices with multiple files: {len(multiple_files)}")

print("\nPairing analysis completed.")

In [ ]:
import os
import hashlib
from collections import defaultdict
from PIL import Image
from tqdm.auto import tqdm

MENDELEY_ROOT = (
    "/kaggle/input/datasets/prishabhkumar/"
    "mendely-dataset/Mendely"
)

FOLDERS = {
    "Primary": os.path.join(MENDELEY_ROOT, "Primary Image"),
    "Cropped": os.path.join(MENDELEY_ROOT, "Cropped Image")
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ---------------------------------------------------------
# STEP 1: COLLECT FILES WITH MULTIPLE EXTENSIONS
# ---------------------------------------------------------

print("=" * 80)
print("STEP 1: COLLECTING MULTI-EXTENSION IMAGE GROUPS")
print("=" * 80)

groups = defaultdict(list)

for representation, folder in FOLDERS.items():

    for root, dirs, files in os.walk(folder):

        class_name = os.path.basename(root).lower()

        for filename in files:

            if not filename.lower().endswith(IMAGE_EXTENSIONS):
                continue

            import re
            match = re.search(r"\(\s*(\d+)\s*\)", filename)

            if not match:
                continue

            index = int(match.group(1))

            key = (representation, class_name, index)

            groups[key].append(os.path.join(root, filename))

multi_groups = {
    key: paths for key, paths in groups.items()
    if len(paths) > 1
}

print(f"Multiple-extension groups: {len(multi_groups)}")

# ---------------------------------------------------------
# STEP 2: HASH AND COMPARE EACH FILE
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 2: COMPARING IMAGE CONTENT")
print("=" * 80)

for key, paths in tqdm(
    multi_groups.items(),
    desc="Comparing image versions",
    unit="group"
):

    representation, class_name, index = key

    print(f"\n{representation}/{class_name} - Index {index}")

    hashes = []
    dimensions = []

    for path in paths:

        try:
            with Image.open(path) as img:

                dimensions.append(img.size)

                digest = hashlib.sha256(
                    img.convert("RGB").tobytes()
                ).hexdigest()

                hashes.append(digest)

            print(f"  File: {os.path.basename(path)}")
            print(f"  Dimensions: {dimensions[-1]}")
            print(f"  Hash: {digest[:16]}")

        except Exception as e:
            print(f"  Error reading {path}: {e}")

    if len(set(hashes)) == 1 and len(hashes) == len(paths):
        print("  RESULT: IDENTICAL PIXEL CONTENT")
    else:
        print("  RESULT: DIFFERENT PIXEL CONTENT")

# ---------------------------------------------------------
# STEP 3: SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("MULTI-EXTENSION AUDIT COMPLETED")
print("=" * 80)

In [ ]:
import os
import re
import cv2
import numpy as np
from PIL import Image
from collections import defaultdict
from itertools import combinations
from tqdm.auto import tqdm

MENDELEY_ROOT = (
    "/kaggle/input/datasets/prishabhkumar/"
    "mendely-dataset/Mendely"
)

FOLDERS = {
    "Primary": os.path.join(MENDELEY_ROOT, "Primary Image"),
    "Cropped": os.path.join(MENDELEY_ROOT, "Cropped Image")
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ---------------------------------------------------------
# STEP 1: COLLECT FILES BY CLASS AND INDEX
# ---------------------------------------------------------

print("=" * 80)
print("STEP 1: COLLECTING IMAGE GROUPS")
print("=" * 80)

image_groups = defaultdict(list)

for representation, folder in FOLDERS.items():

    for root, dirs, files in os.walk(folder):

        class_name = os.path.basename(root).lower()

        for filename in files:

            if not filename.lower().endswith(IMAGE_EXTENSIONS):
                continue

            match = re.search(r"\(\s*(\d+)\s*\)", filename)

            if match:
                index = int(match.group(1))

                image_groups[(class_name, index)].append({
                    "representation": representation,
                    "filename": filename,
                    "path": os.path.join(root, filename)
                })

print(f"Total class-index groups: {len(image_groups)}")

# ---------------------------------------------------------
# STEP 2: DEFINE PERCEPTUAL HASH
# ---------------------------------------------------------

def calculate_phash(image_path):

    with Image.open(image_path) as img:
        gray = img.convert("L").resize((32, 32))

    pixels = np.asarray(gray, dtype=np.float32)

    dct = cv2.dct(pixels)

    low_frequency = dct[:8, :8].flatten()

    median_value = np.median(low_frequency[1:])

    return low_frequency > median_value


def hamming_distance(hash1, hash2):
    return int(np.count_nonzero(hash1 != hash2))


# ---------------------------------------------------------
# STEP 3: CALCULATE HASHES
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 2: CALCULATING PERCEPTUAL HASHES")
print("=" * 80)

hash_cache = {}
failed_images = []

all_records = [
    record
    for records in image_groups.values()
    for record in records
]

for record in tqdm(
    all_records,
    desc="Calculating perceptual hashes",
    unit="image"
):

    try:
        hash_cache[record["path"]] = calculate_phash(record["path"])

    except Exception as e:
        failed_images.append((record["path"], str(e)))

# ---------------------------------------------------------
# STEP 4: COMPARE SAME-INDEX IMAGE VERSIONS
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 3: PERCEPTUAL SIMILARITY RESULTS")
print("=" * 80)

comparison_results = []

for key, records in tqdm(
    image_groups.items(),
    desc="Comparing image groups",
    unit="group"
):

    valid_records = [
        r for r in records if r["path"] in hash_cache
    ]

    if len(valid_records) < 2:
        continue

    comparisons = []

    for r1, r2 in combinations(valid_records, 2):

        distance = hamming_distance(
            hash_cache[r1["path"]],
            hash_cache[r2["path"]]
        )

        comparisons.append({
            "file1": r1,
            "file2": r2,
            "distance": distance
        })

    comparisons.sort(key=lambda x: x["distance"])

    comparison_results.append({
        "key": key,
        "comparisons": comparisons
    })

# ---------------------------------------------------------
# STEP 5: DISPLAY RESULTS
# ---------------------------------------------------------

for result in comparison_results:

    class_name, index = result["key"]

    print(f"\n{class_name.upper()} - INDEX {index}")

    for comparison in result["comparisons"]:

        r1 = comparison["file1"]
        r2 = comparison["file2"]
        distance = comparison["distance"]

        if distance <= 10:
            status = "VERY SIMILAR"
        elif distance <= 20:
            status = "POSSIBLY SIMILAR"
        else:
            status = "LOW SIMILARITY"

        print(
            f"  {r1['representation']}/{r1['filename']}"
            f" <-> {r2['representation']}/{r2['filename']}"
        )

        print(f"  pHash distance: {distance}/64 | {status}")

# ---------------------------------------------------------
# STEP 6: FINAL SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("PERCEPTUAL HASH AUDIT SUMMARY")
print("=" * 80)

print(f"Total image files       : {len(all_records)}")
print(f"Successfully hashed    : {len(hash_cache)}")
print(f"Failed images          : {len(failed_images)}")
print(f"Groups compared        : {len(comparison_results)}")
print(f"Total pair comparisons : {sum(len(r['comparisons']) for r in comparison_results)}")

print("\nAudit completed.")

# Summarize perceptual similarity results

from collections import Counter

all_comparisons = [
    comparison
    for result in comparison_results
    for comparison in result["comparisons"]
]

print("=" * 65)
print("PERCEPTUAL HASH AUDIT SUMMARY")
print("=" * 65)

print(f"Total class-index groups compared : {len(comparison_results)}")
print(f"Total pairwise comparisons        : {len(all_comparisons)}")
print(f"Failed image hashes               : {len(failed_images)}")

distance_counts = Counter()

for item in all_comparisons:
    distance = item["distance"]

    if distance <= 5:
        category = "Near-identical (0-5)"
    elif distance <= 10:
        category = "Very similar (6-10)"
    elif distance <= 20:
        category = "Possibly similar (11-20)"
    else:
        category = "Low similarity (21-64)"

    distance_counts[category] += 1

for category in [
    "Near-identical (0-5)",
    "Very similar (6-10)",
    "Possibly similar (11-20)",
    "Low similarity (21-64)"
]:
    print(f"{category:30s}: {distance_counts[category]}")

print("\n" + "=" * 65)
print("CLOSE MATCHES (DISTANCE <= 20)")
print("=" * 65)

close_matches = [
    item for item in all_comparisons
    if item["distance"] <= 20
]

for item in close_matches:
    r1 = item["file1"]
    r2 = item["file2"]

    print(
        f"{item['distance']:2d}/64 | "
        f"{r1['representation']}/{r1['filename']} <-> "
        f"{r2['representation']}/{r2['filename']}"
    )

print("\nTotal close matches:", len(close_matches))

In [ ]:
from PIL import Image
from collections import defaultdict, Counter
from tqdm.auto import tqdm
import os

dimension_stats = defaultdict(Counter)

for representation, folder in FOLDERS.items():
    image_paths = []

    for root, dirs, files in os.walk(folder):
        for filename in files:
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                image_paths.append((
                    os.path.basename(root),
                    os.path.join(root, filename)
                ))

    print(f"\nScanning {representation}: {len(image_paths)} images")

    for class_name, path in tqdm(
        image_paths,
        desc=f"Reading {representation} dimensions",
        unit="image"
    ):
        try:
            with Image.open(path) as img:
                dimension_stats[(representation, class_name)][img.size] += 1
        except Exception:
            pass

print("\n" + "=" * 75)
print("IMAGE DIMENSION DISTRIBUTION")
print("=" * 75)

for key, dimensions in sorted(dimension_stats.items()):
    representation, class_name = key

    print(f"\n{representation} / {class_name}")
    print(f"Unique dimensions: {len(dimensions)}")

    for size, count in dimensions.most_common(10):
        print(f"  {size[0]} x {size[1]} : {count}")

    if len(dimensions) > 10:
        print(f"  ... and {len(dimensions) - 10} other dimension patterns")

In [ ]:
from PIL import Image
from collections import Counter, defaultdict
import os
from tqdm.auto import tqdm

metadata_summary = defaultdict(Counter)
sample_metadata = defaultdict(list)

for representation, folder in FOLDERS.items():

    image_paths = []

    for root, dirs, files in os.walk(folder):
        for filename in files:
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                image_paths.append((
                    os.path.basename(root),
                    os.path.join(root, filename)
                ))

    print(f"\nInspecting {representation}: {len(image_paths)} images")

    for class_name, path in tqdm(
        image_paths,
        desc=f"Reading {representation} metadata",
        unit="image"
    ):
        try:
            with Image.open(path) as img:

                exif = img.getexif()
                metadata_summary[representation]["total"] += 1

                if exif:
                    metadata_summary[representation]["with_exif"] += 1
                else:
                    metadata_summary[representation]["without_exif"] += 1

                if len(sample_metadata[representation]) < 5:
                    sample_metadata[representation].append({
                        "class": class_name,
                        "filename": os.path.basename(path),
                        "format": img.format,
                        "mode": img.mode,
                        "exif_tags": {
                            str(tag): str(value)[:100]
                            for tag, value in exif.items()
                        }
                    })

        except Exception:
            metadata_summary[representation]["unreadable"] += 1

print("\n" + "=" * 70)
print("METADATA SUMMARY")
print("=" * 70)

for representation, counts in metadata_summary.items():
    print(f"\n{representation}")
    for key, value in counts.items():
        print(f"  {key}: {value}")

    print("\nSample metadata:")
    for sample in sample_metadata[representation]:
        print(sample)

In [ ]:
from PIL import Image
from collections import Counter
from tqdm.auto import tqdm
import os

exif_summary = {
    "Primary": {
        "camera_make": Counter(),
        "camera_model": Counter(),
        "software": Counter(),
        "date_prefix": Counter()
    },
    "Cropped": {
        "camera_make": Counter(),
        "camera_model": Counter(),
        "software": Counter(),
        "date_prefix": Counter()
    }
}

tag_names = {
    271: "camera_make",
    272: "camera_model",
    305: "software",
    306: "datetime"
}

for representation, folder in FOLDERS.items():

    image_paths = []

    for root, dirs, files in os.walk(folder):
        for filename in files:
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                image_paths.append(os.path.join(root, filename))

    for path in tqdm(
        image_paths,
        desc=f"Analyzing {representation} EXIF",
        unit="image"
    ):
        try:
            with Image.open(path) as img:
                exif = img.getexif()

                values = {}

                for tag_id, key in tag_names.items():
                    value = exif.get(tag_id)

                    if value:
                        values[key] = str(value).strip().replace("\x00", "")

                for key in ["camera_make", "camera_model", "software"]:
                    if values.get(key):
                        exif_summary[representation][key][values[key]] += 1

                if values.get("datetime"):
                    date = values["datetime"][:10]
                    exif_summary[representation]["date_prefix"][date] += 1

        except Exception:
            pass

for representation, categories in exif_summary.items():

    print("\n" + "=" * 70)
    print(representation.upper(), "EXIF DISTRIBUTION")
    print("=" * 70)

    for category, counter in categories.items():
        print(f"\n{category.upper()}")

        if not counter:
            print("  No values found")
            continue

        for value, count in counter.most_common(15):
            print(f"  {value}: {count}")

        if len(counter) > 15:
            print(f"  ... {len(counter) - 15} additional values")

In [ ]:
from PIL import Image
from collections import defaultdict, Counter
from tqdm.auto import tqdm
import os

primary_root = FOLDERS["Primary"]

class_metadata = defaultdict(
    lambda: {
        "camera": Counter(),
        "date": Counter(),
        "total": 0,
        "camera_missing": 0,
        "date_missing": 0
    }
)

image_paths = []

for root, dirs, files in os.walk(primary_root):
    class_name = os.path.basename(root)

    for filename in files:
        if filename.lower().endswith(IMAGE_EXTENSIONS):
            image_paths.append(
                (class_name, os.path.join(root, filename))
            )

for class_name, path in tqdm(
    image_paths,
    desc="Analyzing metadata by emotion",
    unit="image"
):
    try:
        with Image.open(path) as img:
            exif = img.getexif()

            make = str(exif.get(271, "")).strip()
            model = str(exif.get(272, "")).strip()
            date = str(exif.get(306, "")).strip()

            stats = class_metadata[class_name]
            stats["total"] += 1

            if make and make != "--":
                stats["camera"][make] += 1
            else:
                stats["camera_missing"] += 1

            if date:
                stats["date"][date[:10]] += 1
            else:
                stats["date_missing"] += 1

    except Exception:
        pass

print("\n" + "=" * 75)
print("CAMERA AND COLLECTION DATE DISTRIBUTION BY CLASS")
print("=" * 75)

for class_name in sorted(class_metadata):
    stats = class_metadata[class_name]

    print(f"\nCLASS: {class_name.upper()}")
    print(f"Total images: {stats['total']}")
    print(f"Camera metadata missing: {stats['camera_missing']}")
    print(f"Date metadata missing: {stats['date_missing']}")

    print("Camera makes:")
    for camera, count in stats["camera"].most_common():
        print(f"  {camera}: {count}")

    print("Collection dates:")
    for date, count in stats["date"].most_common(10):
        print(f"  {date}: {count}")

## Build the complete date × emotion and camera × emotion tables

In [ ]:
import pandas as pd

classes = sorted(class_metadata.keys())

# --------------------------------------------------
# 1. COLLECTION DATE VS EMOTION
# --------------------------------------------------

all_dates = sorted({
    date
    for stats in class_metadata.values()
    for date in stats["date"]
})

date_table = pd.DataFrame(
    0,
    index=all_dates,
    columns=classes
)

for class_name, stats in class_metadata.items():
    for date, count in stats["date"].items():
        date_table.loc[date, class_name] = count

print("=" * 85)
print("COLLECTION DATE × EMOTION")
print("=" * 85)
print(date_table.to_string())

print("\nPercentage distribution within each emotion:")
print(
    (date_table.div(date_table.sum(axis=0), axis=1) * 100)
    .round(1)
    .to_string()
)

# --------------------------------------------------
# 2. CAMERA MAKE VS EMOTION
# --------------------------------------------------

all_cameras = sorted({
    camera
    for stats in class_metadata.values()
    for camera in stats["camera"]
})

camera_table = pd.DataFrame(
    0,
    index=all_cameras,
    columns=classes
)

for class_name, stats in class_metadata.items():
    for camera, count in stats["camera"].items():
        camera_table.loc[camera, class_name] = count

print("\n" + "=" * 85)
print("CAMERA MAKE × EMOTION")
print("=" * 85)
print(camera_table.to_string())

print("\nPercentage distribution within each emotion:")
print(
    (camera_table.div(camera_table.sum(axis=0), axis=1) * 100)
    .round(1)
    .to_string()
)

In [ ]:
import os
from PIL import Image
from collections import defaultdict, Counter
from datetime import datetime
from tqdm.auto import tqdm

timestamp_records = []

primary_root = FOLDERS["Primary"]

image_paths = []

for root, dirs, files in os.walk(primary_root):
    class_name = os.path.basename(root)

    for filename in files:
        if filename.lower().endswith(IMAGE_EXTENSIONS):
            image_paths.append({
                "class": class_name,
                "filename": filename,
                "path": os.path.join(root, filename)
            })

for record in tqdm(
    image_paths,
    desc="Extracting original capture timestamps",
    unit="image"
):
    try:
        with Image.open(record["path"]) as img:
            exif = img.getexif()

            # Prefer DateTimeOriginal; fall back to DateTime.
            exif_ifd = exif.get_ifd(34665)

            raw_timestamp = (
                exif_ifd.get(36867)
                or exif.get(306)
            )

            if raw_timestamp:
                timestamp = datetime.strptime(
                    str(raw_timestamp).strip(),
                    "%Y:%m:%d %H:%M:%S"
                )

                timestamp_records.append({
                    "class": record["class"],
                    "filename": record["filename"],
                    "timestamp": timestamp,
                    "camera": str(exif.get(271, "Unknown")).strip()
                })

    except Exception:
        pass

# Sort all available timestamps.
timestamp_records.sort(key=lambda x: x["timestamp"])

print("\n" + "=" * 75)
print("CAPTURE TIMESTAMP AUDIT")
print("=" * 75)

print(f"Total Primary images: {len(image_paths)}")
print(f"Images with usable timestamps: {len(timestamp_records)}")

# Count timestamp concentration by date and hour.
date_hour_counts = Counter(
    (r["timestamp"].strftime("%Y-%m-%d"),
     r["timestamp"].strftime("%H"))
    for r in timestamp_records
)

print("\nMost active capture date-hour combinations:")

for (date, hour), count in date_hour_counts.most_common(20):
    print(f"  {date} {hour}:00 - {count} images")

# Count images captured within 60 seconds of the previous image.
short_gap_counts = Counter()

for previous, current in zip(
    timestamp_records,
    timestamp_records[1:]
):
    gap = (
        current["timestamp"] - previous["timestamp"]
    ).total_seconds()

    if gap <= 60:
        short_gap_counts[current["class"]] += 1

print("\nImages following another capture within 60 seconds:")
for class_name, count in sorted(short_gap_counts.items()):
    print(f"  {class_name}: {count}")

# Show a few timestamp examples.
print("\nSample chronological records:")

for record in timestamp_records[:15]:
    print(
        record["timestamp"],
        "|",
        record["class"],
        "|",
        record["camera"],
        "|",
        record["filename"]
    )

In [ ]:
from collections import defaultdict, Counter
import pandas as pd

# Sort chronologically
records = sorted(
    timestamp_records,
    key=lambda x: x["timestamp"]
)

SESSION_GAP_SECONDS = 300  # 5 minutes

sessions = []
current_session = []

for record in records:

    if not current_session:
        current_session = [record]
        continue

    gap = (
        record["timestamp"] -
        current_session[-1]["timestamp"]
    ).total_seconds()

    if gap <= SESSION_GAP_SECONDS:
        current_session.append(record)
    else:
        sessions.append(current_session)
        current_session = [record]

if current_session:
    sessions.append(current_session)

# Summarize sessions
session_rows = []

for session_id, session in enumerate(sessions, start=1):

    class_counts = Counter(
        record["class"] for record in session
    )

    cameras = Counter(
        record["camera"] for record in session
        if record["camera"] and record["camera"] != "--"
    )

    session_rows.append({
        "session_id": session_id,
        "date": session[0]["timestamp"].date(),
        "start": session[0]["timestamp"],
        "end": session[-1]["timestamp"],
        "images": len(session),
        "classes": len(class_counts),
        "emotion_counts": dict(class_counts),
        "cameras": dict(cameras)
    })

session_df = pd.DataFrame(session_rows)

print("=" * 75)
print("CAPTURE SESSION AUDIT (5-MINUTE GAP)")
print("=" * 75)

print(f"Timestamped images: {len(records)}")
print(f"Identified sessions: {len(sessions)}")
print(f"Singleton sessions: {(session_df['images'] == 1).sum()}")
print(f"Sessions with 2+ images: {(session_df['images'] >= 2).sum()}")
print(f"Sessions with 10+ images: {(session_df['images'] >= 10).sum()}")
print(f"Largest session: {session_df['images'].max()} images")

print("\nSESSION SIZE DISTRIBUTION:")
print(
    session_df["images"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90])
    .round(2)
    .to_string()
)

print("\nSESSIONS WITH THE MOST IMAGES:")
print(
    session_df.nlargest(20, "images")[
        ["session_id", "date", "start", "end",
         "images", "classes", "emotion_counts"]
    ].to_string(index=False)
)

print("\nCLASS COMPOSITION:")
print(
    session_df["classes"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nSESSIONS CONTAINING MULTIPLE EMOTIONS:")
multi_class = session_df[session_df["classes"] > 1]

print(f"Count: {len(multi_class)}")

print(
    multi_class.nlargest(15, "images")[
        ["session_id", "date", "start",
         "images", "emotion_counts"]
    ].to_string(index=False)
)

In [ ]:
import os
import hashlib
from collections import defaultdict, Counter
from PIL import Image
from tqdm.auto import tqdm

primary_root = FOLDERS["Primary"]

primary_records = []

for root, dirs, files in os.walk(primary_root):
    class_name = os.path.basename(root)

    for filename in files:
        if filename.lower().endswith(IMAGE_EXTENSIONS):
            primary_records.append({
                "class": class_name,
                "filename": filename,
                "path": os.path.join(root, filename)
            })

hash_groups = defaultdict(list)
failed = []

print(f"Primary images discovered: {len(primary_records)}")

for record in tqdm(
    primary_records,
    desc="Hashing Primary images",
    unit="image"
):
    try:
        with Image.open(record["path"]) as img:
            rgb = img.convert("RGB")
            image_hash = hashlib.sha256(
                str(rgb.size).encode() + rgb.tobytes()
            ).hexdigest()

        hash_groups[image_hash].append(record)

    except Exception as e:
        failed.append((record["path"], str(e)))

duplicate_groups = [
    group for group in hash_groups.values()
    if len(group) > 1
]

same_class_groups = []
cross_class_groups = []

for group in duplicate_groups:
    classes = {record["class"] for record in group}

    if len(classes) > 1:
        cross_class_groups.append(group)
    else:
        same_class_groups.append(group)

print("\n" + "=" * 70)
print("PRIMARY EXACT-DUPLICATE AUDIT")
print("=" * 70)

print(f"Images discovered       : {len(primary_records)}")
print(f"Successfully hashed     : {len(primary_records) - len(failed)}")
print(f"Unique image hashes     : {len(hash_groups)}")
print(f"Duplicate groups        : {len(duplicate_groups)}")
print(f"Same-class groups       : {len(same_class_groups)}")
print(f"Cross-class groups      : {len(cross_class_groups)}")
print(f"Redundant copies        : {sum(len(g)-1 for g in duplicate_groups)}")
print(f"Images in duplicate groups: {sum(len(g) for g in duplicate_groups)}")
print(f"Failed images           : {len(failed)}")

print("\n" + "=" * 70)
print("DUPLICATE GROUP DETAILS")
print("=" * 70)

for group in duplicate_groups[:50]:
    classes = {r["class"] for r in group}
    label = "CROSS-CLASS" if len(classes) > 1 else "SAME-CLASS"

    print(f"\n[{label}]")
    for record in group:
        print(f"  {record['class']}/{record['filename']}")

if len(duplicate_groups) > 50:
    print(f"\nOnly first 50 of {len(duplicate_groups)} groups displayed.")

## Calculate the clean Primary distribution

In [ ]:
from collections import Counter

primary_clean_records = []
primary_conflict_records = []

for group in hash_groups.values():

    classes = {record["class"] for record in group}

    if len(classes) > 1:
        # Exclude every member of cross-class duplicate groups.
        primary_conflict_records.extend(group)

    else:
        # Retain only one copy of same-class duplicates.
        primary_clean_records.append(group[0])

primary_counts = Counter(
    record["class"] for record in primary_clean_records
)

conflict_counts = Counter(
    record["class"] for record in primary_conflict_records
)

print("=" * 65)
print("MENDELEY PRIMARY: CLEAN DATASET SUMMARY")
print("=" * 65)

print("\nClean images per class:")
for class_name, count in sorted(primary_counts.items()):
    print(f"{class_name:<15}: {count}")

print(f"\nTotal original images : {len(primary_records)}")
print(f"Clean images          : {len(primary_clean_records)}")
print(f"Conflict images       : {len(primary_conflict_records)}")
print(f"Images excluded       : {len(primary_records) - len(primary_clean_records)}")

print("\nImages involved in cross-class conflicts:")
for class_name, count in sorted(conflict_counts.items()):
    print(f"{class_name:<15}: {count}")

print("\nNote: No physical files have been deleted.")

## Check duplicates across all four sources together

In [ ]:
import os
import hashlib
from collections import defaultdict
from PIL import Image
from tqdm.auto import tqdm

DATA_SOURCES = {
    "Dataset1": os.path.join(
        "/kaggle/input/datasets/prishabhkumar/dataset1/dataset",
        "train"
    ),
    "Dataset2": os.path.join(
        "/kaggle/input/datasets/prishabhkumar/dataset2/dataset",
        "train"
    ),
    "Mendeley_Cropped": "/kaggle/input/datasets/prishabhkumar/mendely-dataset/Mendely/Cropped Image",
    "Mendeley_Primary": "/kaggle/input/datasets/prishabhkumar/mendely-dataset/Mendely/Primary Image"
}

all_records = []

for source, root in DATA_SOURCES.items():
    for class_name in sorted(os.listdir(root)):
        class_path = os.path.join(root, class_name)

        if not os.path.isdir(class_path):
            continue

        for filename in os.listdir(class_path):
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                all_records.append({
                    "source": source,
                    "class": class_name,
                    "filename": filename,
                    "path": os.path.join(class_path, filename)
                })

global_hash_groups = defaultdict(list)
failed = []

print(f"Total images discovered: {len(all_records)}")

for record in tqdm(
    all_records,
    desc="Final cross-source duplicate audit",
    unit="image"
):
    try:
        with Image.open(record["path"]) as img:
            rgb = img.convert("RGB")
            digest = hashlib.sha256(
                str(rgb.size).encode() + rgb.tobytes()
            ).hexdigest()

        global_hash_groups[digest].append(record)

    except Exception as e:
        failed.append((record["path"], str(e)))

duplicate_groups = [
    group for group in global_hash_groups.values()
    if len(group) > 1
]

cross_source = [
    group for group in duplicate_groups
    if len({r["source"] for r in group}) > 1
]

cross_class = [
    group for group in duplicate_groups
    if len({r["class"] for r in group}) > 1
]

print("\n" + "=" * 70)
print("FINAL GLOBAL DUPLICATE AUDIT")
print("=" * 70)

print(f"Images discovered          : {len(all_records)}")
print(f"Successfully hashed        : {len(all_records)-len(failed)}")
print(f"Unique image hashes        : {len(global_hash_groups)}")
print(f"Total duplicate groups     : {len(duplicate_groups)}")
print(f"Cross-source groups        : {len(cross_source)}")
print(f"Cross-class groups         : {len(cross_class)}")
print(f"Redundant copies           : {sum(len(g)-1 for g in duplicate_groups)}")
print(f"Failed images              : {len(failed)}")

print("\nCROSS-SOURCE DUPLICATE DETAILS")

for group in cross_source:
    print("\nDuplicate group:")
    for r in group:
        print(f"  {r['source']}/{r['class']}/{r['filename']}")

## Build the final clean manifest

In [ ]:
import os
import hashlib
from collections import defaultdict, Counter
from PIL import Image
from tqdm.auto import tqdm

DATA_SOURCES = {
    "Dataset1": "/kaggle/input/datasets/prishabhkumar/dataset1/dataset/train",
    "Dataset2": "/kaggle/input/datasets/prishabhkumar/dataset2/dataset/train",
    "Mendeley_Cropped": "/kaggle/input/datasets/prishabhkumar/mendely-dataset/Mendely/Cropped Image",
    "Mendeley_Primary": "/kaggle/input/datasets/prishabhkumar/mendely-dataset/Mendely/Primary Image"
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

all_records = []

for source, root in DATA_SOURCES.items():
    for class_name in sorted(os.listdir(root)):
        class_path = os.path.join(root, class_name)

        if not os.path.isdir(class_path):
            continue

        for filename in os.listdir(class_path):
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                all_records.append({
                    "source": source,
                    "class": class_name,
                    "filename": filename,
                    "path": os.path.join(class_path, filename)
                })

print(f"Images discovered: {len(all_records)}")

global_hash_groups = defaultdict(list)
failed = []

for record in tqdm(all_records, desc="Hashing images", unit="image"):
    try:
        with Image.open(record["path"]) as img:
            rgb = img.convert("RGB")
            digest = hashlib.sha256(
                str(rgb.size).encode() + rgb.tobytes()
            ).hexdigest()

        global_hash_groups[digest].append(record)

    except Exception as e:
        failed.append((record["path"], str(e)))

final_records = []
excluded_conflicts = []
removed_redundant = []

for group in global_hash_groups.values():

    labels = {r["class"].strip().lower() for r in group}

    if len(labels) > 1:
        excluded_conflicts.extend(group)
    else:
        final_records.append(group[0])
        removed_redundant.extend(group[1:])

source_class_counts = defaultdict(Counter)

for record in final_records:
    source_class_counts[record["source"]][record["class"]] += 1

print("\n" + "=" * 70)
print("FINAL CLEAN DATASET MANIFEST")
print("=" * 70)

for source in DATA_SOURCES:
    print(f"\n{source}")

    for class_name, count in sorted(source_class_counts[source].items()):
        print(f"  {class_name:<15}: {count}")

    print(f"  {'TOTAL':<15}: {sum(source_class_counts[source].values())}")

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"Original images          : {len(all_records)}")
print(f"Unique hashes            : {len(global_hash_groups)}")
print(f"Clean images             : {len(final_records)}")
print(f"Cross-class exclusions   : {len(excluded_conflicts)}")
print(f"Redundant copies removed : {len(removed_redundant)}")
print(f"Failed images            : {len(failed)}")

print("\nNo physical files have been deleted.")

## Create the dataset splits

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

RANDOM_STATE = 42

# --------------------------------------------------
# 1. Prepare records for the two classification heads
# --------------------------------------------------

head1_records = [
    {
        "task": "seven_class",
        "source": r["source"],
        "label": r["class"].lower(),
        "path": r["path"]
    }
    for r in final_records
    if r["source"] in ["Dataset1", "Dataset2"]
]

head2_records = [
    {
        "task": "infant_expression",
        "source": r["source"],
        "label": r["class"].lower(),
        "path": r["path"]
    }
    for r in final_records
    if r["source"] == "Mendeley_Cropped"
]

# --------------------------------------------------
# 2. Define stratified 70:15:15 splitting function
# --------------------------------------------------

def create_stratified_splits(records, task_name):

    labels = [r["label"] for r in records]

    train_records, temp_records = train_test_split(
        records,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=labels
    )

    temp_labels = [r["label"] for r in temp_records]

    val_records, test_records = train_test_split(
        temp_records,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=temp_labels
    )

    for split_name, split_records in [
        ("train", train_records),
        ("validation", val_records),
        ("test", test_records)
    ]:
        for record in split_records:
            record["split"] = split_name

    print(f"\n{task_name}")
    print("-" * 55)

    for split_name, split_records in [
        ("Train", train_records),
        ("Validation", val_records),
        ("Test", test_records)
    ]:
        counts = Counter(r["label"] for r in split_records)

        print(f"\n{split_name}: {len(split_records)} images")
        for label, count in sorted(counts.items()):
            print(f"  {label:<12}: {count}")

    return train_records + val_records + test_records


# --------------------------------------------------
# 3. Generate both independent task splits
# --------------------------------------------------

print("=" * 65)
print("STRATIFIED DATASET SPLITTING")
print("=" * 65)

head1_splits = create_stratified_splits(
    head1_records,
    "HEAD 1 — SEVEN-CLASS CLASSIFICATION"
)

head2_splits = create_stratified_splits(
    head2_records,
    "HEAD 2 — INFANT EXPRESSION CLASSIFICATION"
)

# --------------------------------------------------
# 4. Combine and save the split manifest
# --------------------------------------------------

split_records = head1_splits + head2_splits

split_df = pd.DataFrame(split_records)

split_df.to_csv(
    "/kaggle/working/infant_expression_splits.csv",
    index=False
)

print("\n" + "=" * 65)
print("FINAL SPLIT SUMMARY")
print("=" * 65)

print(f"Head 1 images : {len(head1_splits)}")
print(f"Head 2 images : {len(head2_splits)}")
print(f"Total images  : {len(split_df)}")

print("\nSaved manifest:")
print("/kaggle/working/infant_expression_splits.csv")

## Create datasets and data loaders

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# Read the saved split manifest so this cell is
# independent of the previous cell's DataFrame.
split_df = pd.read_csv(
    "/kaggle/working/infant_expression_splits.csv"
)

CLASS_MAPPINGS = {
    "seven_class": {
        "angry": 0,
        "disgust": 1,
        "fear": 2,
        "happy": 3,
        "neutral": 4,
        "sad": 5,
        "surprise": 6
    },
    "infant_expression": {
        "angry": 0,
        "cry": 1,
        "laugh": 2,
        "normal": 3
    }
}

# --------------------------------------------------
# 2. Image transformations
# --------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.08
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --------------------------------------------------
# 3. Custom PyTorch Dataset
# --------------------------------------------------

class ExpressionDataset(Dataset):

    def __init__(self, dataframe, task, transform=None):
        self.data = dataframe[
            (dataframe["task"] == task)
        ].reset_index(drop=True)

        self.task = task
        self.transform = transform
        self.class_mapping = CLASS_MAPPINGS[task]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        record = self.data.iloc[index]

        image = Image.open(record["path"]).convert("RGB")
        label = self.class_mapping[record["label"]]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


# --------------------------------------------------
# 4. Create separate loaders for both tasks
# --------------------------------------------------

dataloaders = {}

for task in ["seven_class", "infant_expression"]:

    dataloaders[task] = {}

    for split in ["train", "validation", "test"]:

        subset = split_df[
            (split_df["task"] == task) &
            (split_df["split"] == split)
        ]

        transform = (
            train_transform if split == "train"
            else eval_transform
        )

        dataset = ExpressionDataset(
            dataframe=subset,
            task=task,
            transform=transform
        )

        dataloaders[task][split] = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=(split == "train"),
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available()
        )

# --------------------------------------------------
# 5. Verify batches
# --------------------------------------------------

print("\n" + "=" * 65)
print("DATALOADER VERIFICATION")
print("=" * 65)

for task in dataloaders:

    print(f"\nTask: {task}")

    for split in dataloaders[task]:

        images, labels = next(iter(dataloaders[task][split]))

        print(
            f"{split:<12} | "
            f"Images: {tuple(images.shape)} | "
            f"Labels: {tuple(labels.shape)} | "
            f"Classes: {labels.unique().tolist()}"
        )

print("\nClass mappings:")
for task, mapping in CLASS_MAPPINGS.items():
    print(f"{task}: {mapping}")

print("\nData loaders created successfully.")

## Shared CNN backbone with two classification heads
- A shared feature extractor.
- A seven-class classification head.
- A four-class infant-expression head.
- Global average pooling and dropout.

In [ ]:
import torch
import torch.nn as nn

# --------------------------------------------------
# 1. Depthwise-Separable Convolution Block
# --------------------------------------------------

class DepthwiseSeparableBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.block = nn.Sequential(

            # Depthwise convolution
            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_channels,
                bias=False
            ),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),

            # Pointwise convolution
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# --------------------------------------------------
# 2. Shared CNN Backbone + Two Classification Heads
# --------------------------------------------------

class MultiTaskInfantCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Shared feature extractor
        self.backbone = nn.Sequential(

            # Initial feature extraction
            nn.Conv2d(
                3, 32,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True),

            # Progressive feature extraction
            DepthwiseSeparableBlock(32, 64, stride=2),
            DepthwiseSeparableBlock(64, 96, stride=2),
            DepthwiseSeparableBlock(96, 160, stride=2),
            DepthwiseSeparableBlock(160, 192, stride=2),

            # Global spatial aggregation
            nn.AdaptiveAvgPool2d(1)
        )

        # Shared classifier input
        self.dropout = nn.Dropout(p=0.35)

        # Head 1: Seven-class classification
        self.seven_class_head = nn.Linear(192, 7)

        # Head 2: Infant-expression classification
        self.infant_expression_head = nn.Linear(192, 4)

    def forward(self, x, task):

        features = self.backbone(x)

        features = torch.flatten(features, 1)
        features = self.dropout(features)

        if task == "seven_class":
            return self.seven_class_head(features)

        elif task == "infant_expression":
            return self.infant_expression_head(features)

        else:
            raise ValueError(f"Unknown task: {task}")


# --------------------------------------------------
# 3. Initialize Model
# --------------------------------------------------

model = MultiTaskInfantCNN().to(device)

# --------------------------------------------------
# 4. Verify Architecture and Output Dimensions
# --------------------------------------------------

model.eval()

with torch.no_grad():

    dummy_input = torch.randn(2, 3, 224, 224).to(device)

    seven_output = model(dummy_input, task="seven_class")

    infant_output = model(
        dummy_input,
        task="infant_expression"
    )

total_parameters = sum(
    parameter.numel() for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("=" * 65)
print("MULTI-TASK CNN ARCHITECTURE VERIFICATION")
print("=" * 65)

print(f"Device                  : {device}")
print(f"Input shape             : {tuple(dummy_input.shape)}")
print(f"Seven-class output      : {tuple(seven_output.shape)}")
print(f"Infant-expression output: {tuple(infant_output.shape)}")

print(f"\nTotal parameters        : {total_parameters:,}")
print(f"Trainable parameters    : {trainable_parameters:,}")

print("\nClassification heads:")
print(f"  Seven-class           : {model.seven_class_head}")
print(f"  Infant-expression     : {model.infant_expression_head}")

print("\nModel initialized successfully.")

## Prepare the training strategy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# --------------------------------------------------
# 1. Loss functions
# --------------------------------------------------

criterion_seven = nn.CrossEntropyLoss()
criterion_infant = nn.CrossEntropyLoss()

# --------------------------------------------------
# 2. Optimizer
# --------------------------------------------------

LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# --------------------------------------------------
# 3. Learning rate scheduler
# --------------------------------------------------

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# --------------------------------------------------
# 4. Training configuration
# --------------------------------------------------

EPOCHS = 30

TASK_WEIGHTS = {
    "seven_class": 0.5,
    "infant_expression": 0.5
}

print("=" * 65)
print("TRAINING CONFIGURATION")
print("=" * 65)

print(f"Device             : {device}")
print(f"Optimizer          : AdamW")
print(f"Learning rate      : {LEARNING_RATE}")
print(f"Weight decay       : {WEIGHT_DECAY}")
print(f"Epochs             : {EPOCHS}")
print(f"Scheduler          : ReduceLROnPlateau")
print(f"Seven-class weight : {TASK_WEIGHTS['seven_class']}")
print(f"Infant weight      : {TASK_WEIGHTS['infant_expression']}")

print("\nLoss functions:")
print(f"Seven-class        : {criterion_seven}")
print(f"Infant-expression  : {criterion_infant}")

print("\nTraining configuration initialized successfully.")

## Identify the DataLoaders

In [ ]:
from torch.utils.data import DataLoader

print("=" * 65)
print("AVAILABLE DATALOADERS")
print("=" * 65)

found = False

for name, obj in list(globals().items()):

    if isinstance(obj, DataLoader):
        print(f"{name}: {len(obj)} batches")
        found = True

    elif isinstance(obj, dict):
        for key, value in obj.items():

            if isinstance(value, DataLoader):
                print(f"{name}[{key}]: {len(value)} batches")
                found = True

            elif isinstance(value, dict):
                for subkey, loader in value.items():
                    if isinstance(loader, DataLoader):
                        print(
                            f"{name}[{key}][{subkey}]: "
                            f"{len(loader)} batches"
                        )
                        found = True

if not found:
    print("No DataLoader objects found in the current session.")

## Complete training and validation loop

In [ ]:

import os
import copy
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Training configuration
# --------------------------------------------------

EPOCHS = 30
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "best_multitask_infant_cnn.pth"
)

history = {
    "train_seven_loss": [],
    "train_seven_acc": [],
    "val_seven_loss": [],
    "val_seven_acc": [],
    "train_infant_loss": [],
    "train_infant_acc": [],
    "val_infant_loss": [],
    "val_infant_acc": []
}

best_infant_val_loss = float("inf")
best_epoch = 0

# --------------------------------------------------
# 2. Single-task training function
# --------------------------------------------------

def train_one_task(model, loader, task, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(
        loader,
        desc=f"Training {task}",
        leave=False
    )

    for images, labels in progress:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images, task=task)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / total, correct / total


# --------------------------------------------------
# 3. Single-task validation function
# --------------------------------------------------

def validate_one_task(model, loader, task, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images, task=task)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)

            running_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += batch_size

    return running_loss / total, correct / total


# --------------------------------------------------
# 4. Complete multi-task training
# --------------------------------------------------

print("=" * 70)
print("STARTING MULTI-TASK CNN TRAINING")
print("=" * 70)

for epoch in range(EPOCHS):

    print(f"\nEPOCH {epoch + 1}/{EPOCHS}")
    print("-" * 70)

    # Train seven-class task
    train_seven_loss, train_seven_acc = train_one_task(
        model,
        dataloaders["seven_class"]["train"],
        "seven_class",
        criterion_seven,
        optimizer
    )

    # Train infant-expression task
    train_infant_loss, train_infant_acc = train_one_task(
        model,
        dataloaders["infant_expression"]["train"],
        "infant_expression",
        criterion_infant,
        optimizer
    )

    # Validate both tasks
    val_seven_loss, val_seven_acc = validate_one_task(
        model,
        dataloaders["seven_class"]["validation"],
        "seven_class",
        criterion_seven
    )

    val_infant_loss, val_infant_acc = validate_one_task(
        model,
        dataloaders["infant_expression"]["validation"],
        "infant_expression",
        criterion_infant
    )

    # Scheduler monitors primary task
    scheduler.step(val_infant_loss)

    # Record history
    history["train_seven_loss"].append(train_seven_loss)
    history["train_seven_acc"].append(train_seven_acc)
    history["val_seven_loss"].append(val_seven_loss)
    history["val_seven_acc"].append(val_seven_acc)

    history["train_infant_loss"].append(train_infant_loss)
    history["train_infant_acc"].append(train_infant_acc)
    history["val_infant_loss"].append(val_infant_loss)
    history["val_infant_acc"].append(val_infant_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"Seven-class:")
    print(f"  Train Loss: {train_seven_loss:.4f} | "
          f"Train Acc: {train_seven_acc:.4f}")
    print(f"  Val Loss  : {val_seven_loss:.4f} | "
          f"Val Acc  : {val_seven_acc:.4f}")

    print(f"\nInfant-expression:")
    print(f"  Train Loss: {train_infant_loss:.4f} | "
          f"Train Acc: {train_infant_acc:.4f}")
    print(f"  Val Loss  : {val_infant_loss:.4f} | "
          f"Val Acc  : {val_infant_acc:.4f}")

    print(f"\nLearning Rate: {current_lr:.7f}")

    # Save best infant-expression checkpoint
    if val_infant_loss < best_infant_val_loss:

        best_infant_val_loss = val_infant_loss
        best_epoch = epoch + 1

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_infant_val_loss": best_infant_val_loss,
            "history": history
        }, BEST_MODEL_PATH)

        print("  Best infant-expression model saved!")

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(f"Best epoch              : {best_epoch}")
print(f"Best infant val loss    : {best_infant_val_loss:.4f}")
print(f"Checkpoint              : {BEST_MODEL_PATH}")

## Detailed validation evaluation

In [ ]:

import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Load the best checkpoint selected using infant-expression validation loss.
checkpoint = torch.load(
    "/kaggle/working/checkpoints/best_multitask_infant_cnn.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

class_names = {
    "seven_class": [
        "angry", "disgust", "fear", "happy",
        "neutral", "sad", "surprise"
    ],
    "infant_expression": [
        "angry", "cry", "laugh", "normal"
    ]
}

def evaluate_validation(task):
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in dataloaders[task]["validation"]:
            images = images.to(device)
            outputs = model(images, task)

            predictions = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predictions.cpu().numpy())

    names = class_names[task]

    print("\n" + "=" * 75)
    print(f"VALIDATION RESULTS: {task.upper()}")
    print("=" * 75)

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=list(range(len(names))),
            target_names=names,
            digits=4,
            zero_division=0
        )
    )

    print("Confusion Matrix:")
    print("Rows = Actual | Columns = Predicted")
    print(names)
    print(confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(names)))
    ))

evaluate_validation("seven_class")
evaluate_validation("infant_expression")

## Inspecting the infant-expression validation predictions from the saved best checkpoint

In [ ]:
import torch
import numpy as np
from collections import Counter

task = "infant_expression"
class_names = ["angry", "cry", "laugh", "normal"]

# Regenerate predictions if the previous cell did not complete
if "all_predictions" not in globals() or "all_labels" not in globals():
    all_predictions = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, labels in dataloaders[task]["validation"]:
            images = images.to(device)
            outputs = model(images, task)

            predictions = torch.argmax(outputs, dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.numpy())

print("=" * 65)
print("INFANT-EXPRESSION VALIDATION PREDICTION DISTRIBUTION")
print("=" * 65)

print("\nActual distribution:")
for class_id, count in sorted(Counter(all_labels).items()):
    print(f"{class_names[class_id]:12s}: {count}")

print("\nPredicted distribution:")
for class_id, count in sorted(Counter(all_predictions).items()):
    print(f"{class_names[class_id]:12s}: {count}")

print("\nTotal validation predictions:", len(all_predictions))

## Inspect the shared backbone's task performance

In [ ]:
import torch

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

criteria = {
    "seven_class": criterion_seven,
    "infant_expression": criterion_infant
}

print("=" * 70)
print("BEST CHECKPOINT: VALIDATION PERFORMANCE")
print("=" * 70)

for task in ["seven_class", "infant_expression"]:

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in dataloaders[task]["validation"]:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images, task)
            loss = criteria[task](outputs, labels)

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (outputs.argmax(1) == labels).sum().item()
            total_samples += batch_size

    print(f"\nTask: {task}")
    print(f"Validation Loss: {total_loss / total_samples:.4f}")
    print(f"Validation Accuracy: {total_correct / total_samples:.4f}")

print("\nCheckpoint epoch:", checkpoint["epoch"])

## Examine task-specific backbone representations

In [ ]:
import torch
import numpy as np

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

task_features = {}

with torch.no_grad():

    for task in ["seven_class", "infant_expression"]:

        features_list = []

        for images, labels in dataloaders[task]["validation"]:

            images = images.to(device)

            features = model.backbone(images)
            features = torch.flatten(features, 1)

            features_list.append(features.cpu().numpy())

        features_array = np.concatenate(features_list, axis=0)

        task_features[task] = features_array

        print("=" * 60)
        print(f"TASK: {task}")
        print("=" * 60)

        print("Feature shape:", features_array.shape)
        print("Mean feature value:", np.mean(features_array))
        print("Feature standard deviation:", np.std(features_array))
        print("Mean feature magnitude:", np.mean(np.linalg.norm(features_array, axis=1)))

# Compare the two task-level feature distributions
seven_mean = task_features["seven_class"].mean(axis=0)
infant_mean = task_features["infant_expression"].mean(axis=0)

cosine_similarity = np.dot(seven_mean, infant_mean) / (
    np.linalg.norm(seven_mean) * np.linalg.norm(infant_mean) + 1e-8
)

print("\n" + "=" * 60)
print("CROSS-TASK FEATURE COMPARISON")
print("=" * 60)
print("Cosine similarity between task mean features:", cosine_similarity)

## Create the infant-only model

## Configure infant-only training

In [10]:
import torch
import torch.nn as nn

class InfantOnlyCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True),

            DepthwiseSeparableBlock(32, 64, stride=2),
            DepthwiseSeparableBlock(64, 96, stride=2),
            DepthwiseSeparableBlock(96, 160, stride=2),
            DepthwiseSeparableBlock(160, 192, stride=2),

            nn.AdaptiveAvgPool2d(1)
        )

        self.dropout = nn.Dropout(0.35)
        self.infant_expression_head = nn.Linear(192, 4)

    def forward(self, x):
        features = self.backbone(x)
        features = torch.flatten(features, 1)
        features = self.dropout(features)

        return self.infant_expression_head(features)


# Create a completely fresh model
infant_only_model = InfantOnlyCNN().to(device)

# Count trainable parameters
total_params = sum(
    p.numel() for p in infant_only_model.parameters()
    if p.requires_grad
)

print("Infant-only CNN created successfully.")
print("Device:", device)
print("Trainable parameters:", f"{total_params:,}")
print("Output classes:", infant_only_model.infant_expression_head.out_features)

NameError: name 'DepthwiseSeparableBlock' is not defined

In [ ]:
import os
import torch.nn as nn
import torch.optim as optim

# Loss function
infant_only_criterion = nn.CrossEntropyLoss()

# Optimizer
infant_only_optimizer = optim.AdamW(
    infant_only_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# Learning-rate scheduler
infant_only_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    infant_only_optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# Separate checkpoint path
INFANT_ONLY_CHECKPOINT_PATH = (
    "/kaggle/working/checkpoints/best_infant_only_cnn.pth"
)

os.makedirs(
    os.path.dirname(INFANT_ONLY_CHECKPOINT_PATH),
    exist_ok=True
)

# Training configuration
INFANT_ONLY_EPOCHS = 30
best_infant_only_val_loss = float("inf")

# Separate training history
infant_only_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rate": []
}

print("Infant-only training configuration ready.")
print("Epochs:", INFANT_ONLY_EPOCHS)
print("Initial learning rate:",
      infant_only_optimizer.param_groups[0]["lr"])
print("Checkpoint:", INFANT_ONLY_CHECKPOINT_PATH)

## Train the infant-only CNN

In [ ]:
from tqdm.auto import tqdm

for epoch in range(INFANT_ONLY_EPOCHS):

    # -------------------- TRAINING --------------------
    infant_only_model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_loader = dataloaders["infant_expression"]["train"]

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{INFANT_ONLY_EPOCHS} [Train]"
    )

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        infant_only_optimizer.zero_grad()

        outputs = infant_only_model(images)
        loss = infant_only_criterion(outputs, labels)

        loss.backward()
        infant_only_optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_accuracy = train_correct / train_total

    # -------------------- VALIDATION --------------------
    infant_only_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    val_loader = dataloaders["infant_expression"]["validation"]

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = infant_only_model(images)
            loss = infant_only_criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # -------------------- SCHEDULER --------------------
    infant_only_scheduler.step(epoch_val_loss)

    current_lr = infant_only_optimizer.param_groups[0]["lr"]

    # -------------------- SAVE BEST MODEL --------------------
    if epoch_val_loss < best_infant_only_val_loss:

        best_infant_only_val_loss = epoch_val_loss

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": infant_only_model.state_dict(),
            "optimizer_state_dict": infant_only_optimizer.state_dict(),
            "best_val_loss": best_infant_only_val_loss,
            "history": infant_only_history
        }, INFANT_ONLY_CHECKPOINT_PATH)

        checkpoint_status = "Saved"

    else:
        checkpoint_status = "-"

    # -------------------- RECORD HISTORY --------------------
    infant_only_history["train_loss"].append(epoch_train_loss)
    infant_only_history["train_accuracy"].append(epoch_train_accuracy)
    infant_only_history["val_loss"].append(epoch_val_loss)
    infant_only_history["val_accuracy"].append(epoch_val_accuracy)
    infant_only_history["learning_rate"].append(current_lr)

    print(
        f"\nEpoch [{epoch + 1:02d}/{INFANT_ONLY_EPOCHS}] "
        f"| Train Loss: {epoch_train_loss:.4f} "
        f"| Train Acc: {epoch_train_accuracy:.4f} "
        f"| Val Loss: {epoch_val_loss:.4f} "
        f"| Val Acc: {epoch_val_accuracy:.4f} "
        f"| LR: {current_lr:.6f} "
        f"| {checkpoint_status}"
    )

print("\n" + "=" * 65)
print("INFANT-ONLY TRAINING COMPLETED")
print("=" * 65)
print("Best validation loss:", round(best_infant_only_val_loss, 4))
print("Checkpoint:", INFANT_ONLY_CHECKPOINT_PATH)

In [ ]:

print("=" * 65)
print("INFANT-ONLY MODEL DIAGNOSTICS")
print("=" * 65)

print("\n1. Model")
print("Infant-only model:", type(infant_only_model).__name__)

shares_parameters = any(
    p1 is p2
    for p1 in infant_only_model.parameters()
    for p2 in model.parameters()
)

print("Shares parameters with multi-task model:", shares_parameters)

print("\n2. Optimizer")
print("Optimizer:", type(infant_only_optimizer).__name__)
print("Learning rate:", infant_only_optimizer.param_groups[0]["lr"])

print("\n3. Scheduler")
print("Scheduler:", type(infant_only_scheduler).__name__)

print("\n4. Checkpoint")
print("Initial best validation loss:", best_infant_only_val_loss)
print("Checkpoint path:", INFANT_ONLY_CHECKPOINT_PATH)

print("\n5. Training configuration")
print("Epochs:", INFANT_ONLY_EPOCHS)

print("\n6. Training history")
print(infant_only_history)

print("\n" + "=" * 65)
print("DIAGNOSTICS COMPLETED")
print("=" * 65)
